# RQ4: does the distribution of UMDA4GI help LLM-guided search?

**Definitions** (the same in every notebook, see the docstring of `analysis.py`):

- *improvement*: % change in CPU cycles of the returned patch on the validation inputs (negative = faster).
  A patch that fails the correctness oracle on the validation inputs counts as 0% (the original program is kept).
- *gap*: for each run, validation change minus training change, in percentage points (positive = the patch
  generalises worse than it trained). Aggregated with the median of the runs.
- *cell*: (software, algorithm, budget), 15 runs = 5 folds × 3 seeds. Each run has 1000 evaluations; the budgets
  100, 250, 500 and 750 are checkpoints of the same run.
- *ranking*: per software, the median of each cell, averaged over the five budgets and ranked; rank sum over the
  four systems (lower = better).

Groups: UMDA4GI without LLM, UMDA4GI whose population is proposed by an LLM (`UMDA-qwen7b`, `UMDA-llama8b`) and the LLM-only algorithm (`LLM-qwen7b`, `LLM-llama8b`).

In [1]:
import pathlib
import sys

import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, str(pathlib.Path('..', 'revision_analysis').resolve()))
import analysis as A

RESULTS = 'results.csv'           # one row per run and budget (checkpoint)
OUT = pathlib.Path('analysis')    # tables written by analysis.py
REFRESH = False                   # True: recompute every table from results.csv
if REFRESH or not (OUT / 'cells.csv').exists():
    A.run(RESULTS, OUT, 'zero')
df = A.load(RESULTS, 'zero')
cells = pd.read_csv(OUT / 'cells.csv')
SW = {'minisat_hack': 'MiniSAT-hack', 'sat4j': 'Sat4J', 'weka': 'Weka', 'pngOptim': 'OptiPNG'}
pd.set_option('display.precision', 2)
pd.set_option('display.max_rows', 300)
pd.set_option('display.width', 200)
HAS_LLM = df.group.isin(A.LLM).any()
print('LLM runs:', df[df.group.isin(A.LLM)].id.nunique())

LLM runs: 0


## Rankings

In [2]:
if HAS_LLM:
    display(pd.read_csv(OUT / 'rq4_improvement_ranking.csv'))
    display(pd.read_csv(OUT / 'rq4_gap_ranking.csv'))

## Friedman and pairwise tests

In [3]:
if HAS_LLM:
    fr = pd.read_csv(OUT / 'rq4_friedman.csv')
    display(fr.pivot_table(index=['metric', 'software'], columns='budget', values='p'))
    pw = pd.read_csv(OUT / 'rq4_pairwise.csv')
    display(pw[pw.p_holm < 0.05])

## How much of the LLM output was used (Reviewer 2, comment 3)

Per group and software, share of the patches asked to the LLM that were malformed, missing (the LLM returned too few) or duplicated (well-formed but already in the population, or empty). Malformed and missing ones are replaced by random 1-edit patches; a duplicated one is replaced by a random patch in UMDA4GI and gets a random edit appended in the LLM-only algorithm. Also the time per LLM call.

In [4]:
if HAS_LLM:
    display(pd.read_csv(OUT / 'rq4_llm_fallback.csv'))

## Diversity of the returned patches

Distinct edited locations over total edits of the patches returned at 1000 evaluations.

In [5]:
if HAS_LLM:
    display(pd.read_csv(OUT / 'diversity.csv'))